# Model Selection

In diesem Notebook werden verschiedene Klassifikationsmodelle für die Human Activity Recognition Data Challenge verglichen.

Die Daten wurden bereits in `02_model_preparation.ipynb` vorbereitet. Für die Modellwahl verwenden wir `train_data.csv`.  
Das finale Testset `test_data.csv` bleibt für die spätere abschließende Bewertung reserviert.

1. Imports und roots (Logistic Regression)

In [1]:
from pathlib import Path
import time
import warnings

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")

In [3]:
PROJECT_ROOT = Path.cwd().parents[1]

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: f:\THB\DataChallenge


In [4]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = PROCESSED_DIR / "train_data.csv"
TEST_PATH = PROCESSED_DIR / "test_data.csv"
FEATURE_COLUMNS_PATH = PROCESSED_DIR / "feature_columns.txt"

print(TRAIN_PATH.exists(), TRAIN_PATH)
print(TEST_PATH.exists(), TEST_PATH)
print(FEATURE_COLUMNS_PATH.exists(), FEATURE_COLUMNS_PATH)

True f:\THB\DataChallenge\data\processed\train_data.csv
True f:\THB\DataChallenge\data\processed\test_data.csv
True f:\THB\DataChallenge\data\processed\feature_columns.txt


2. Daten laden

In [5]:
train_data = pd.read_csv(TRAIN_PATH)
test_data = pd.read_csv(TEST_PATH)

print("train_data:", train_data.shape)
print("test_data:", test_data.shape)

display(train_data.head())
display(test_data.head())

train_data: (5867, 563)
test_data: (1485, 563)


,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,activity
0,0.288585,-0.020294,-0.132905,-0.995279,-0.983111,-0.913526,-0.995112,-0.983185,-0.923527,-0.934724,...,-0.710304,-0.112754,0.030400,-0.464761,-0.018446,-0.841247,0.179941,-0.058627,1.0,STANDING
1,0.278419,-0.016411,-0.123520,-0.998245,-0.975300,-0.960322,-0.998807,-0.974914,-0.957686,-0.943068,...,-0.861499,0.053477,-0.007435,-0.732626,0.703511,-0.844788,0.180289,-0.054317,1.0,STANDING
2,0.279653,-0.019467,-0.113462,-0.995380,-0.967187,-0.978944,-0.996520,-0.963668,-0.977469,-0.938692,...,-0.760104,-0.118559,0.177899,0.100699,0.808529,-0.848933,0.180637,-0.049118,1.0,STANDING
3,0.279174,-0.026201,-0.123283,-0.996091,-0.983403,-0.990675,-0.997099,-0.982750,-0.989302,-0.938692,...,-0.482845,-0.036788,-0.012892,0.640011,-0.485366,-0.848649,0.181935,-0.047663,1.0,STANDING
4,0.276629,-0.016570,-0.115362,-0.998139,-0.980817,-0.990482,-0.998321,-0.979672,-0.990441,-0.942469,...,-0.699205,0.123320,0.122542,0.693578,-0.615971,-0.847865,0.185151,-0.043892,1.0,STANDING


,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,activity
0,0.278791,-0.032477,-0.145829,-0.993050,-0.938822,-0.928840,-0.993505,-0.935597,-0.916592,-0.937860,...,-0.939560,-0.051193,0.102632,0.066183,0.944358,-0.838642,0.209054,0.005301,27.0,STANDING
1,0.275709,-0.017983,-0.102424,-0.995569,-0.981350,-0.978256,-0.995906,-0.981642,-0.981780,-0.942066,...,-0.844675,0.099949,-0.033375,0.629909,0.409820,-0.830250,0.215456,0.014507,27.0,STANDING
2,0.277683,-0.021163,-0.107035,-0.995089,-0.982616,-0.985356,-0.996059,-0.983539,-0.985157,-0.934012,...,-0.818973,-0.108542,0.221648,0.784523,-0.281809,-0.829194,0.216168,0.013870,27.0,STANDING
3,0.277301,-0.015994,-0.098619,-0.994266,-0.978326,-0.976918,-0.995092,-0.978368,-0.975008,-0.934012,...,-0.753174,0.033209,0.391622,0.878145,-0.952204,-0.827296,0.217438,0.012735,27.0,STANDING
4,0.279601,-0.013969,-0.090564,-0.995235,-0.987974,-0.989725,-0.995816,-0.988687,-0.989033,-0.940833,...,-0.921732,-0.013611,-0.501153,0.860905,-0.182502,-0.827697,0.217023,0.009989,27.0,STANDING


In [6]:
print("train columns contain subject:", "subject" in train_data.columns)
print("train columns contain activity:", "activity" in train_data.columns)

print("test columns contain subject:", "subject" in test_data.columns)
print("test columns contain activity:", "activity" in test_data.columns)

print(train_data.columns[:10])
print(train_data.columns[-10:])

train columns contain subject: True
train columns contain activity: True
test columns contain subject: True
test columns contain activity: True
Index(['tBodyAcc-mean()-X', 'tBodyAcc-mean()-Y', 'tBodyAcc-mean()-Z',
       'tBodyAcc-std()-X', 'tBodyAcc-std()-Y', 'tBodyAcc-std()-Z',
       'tBodyAcc-mad()-X', 'tBodyAcc-mad()-Y', 'tBodyAcc-mad()-Z',
       'tBodyAcc-max()-X'],
      dtype='str')
Index(['fBodyBodyGyroJerkMag-kurtosis()', 'angle(tBodyAccMean,gravity)',
       'angle(tBodyAccJerkMean),gravityMean)',
       'angle(tBodyGyroMean,gravityMean)',
       'angle(tBodyGyroJerkMean,gravityMean)', 'angle(X,gravityMean)',
       'angle(Y,gravityMean)', 'angle(Z,gravityMean)', 'subject', 'activity'],
      dtype='str')


In [7]:
with open(FEATURE_COLUMNS_PATH, "r", encoding="utf-8") as file:
    feature_cols = [line.strip() for line in file if line.strip()]

print("Anzahl Features:", len(feature_cols))
print("Erste Features:", feature_cols[:5])
print("Letzte Features:", feature_cols[-5:])

Anzahl Features: 561
Erste Features: ['tBodyAcc-mean()-X', 'tBodyAcc-mean()-Y', 'tBodyAcc-mean()-Z', 'tBodyAcc-std()-X', 'tBodyAcc-std()-Y']
Letzte Features: ['angle(tBodyGyroMean,gravityMean)', 'angle(tBodyGyroJerkMean,gravityMean)', 'angle(X,gravityMean)', 'angle(Y,gravityMean)', 'angle(Z,gravityMean)']


3. Daten Ordnen

In [8]:
# Name der Spalte, die vorhergesagt werden soll
TARGET_COL = "activity"

# Name der Spalte, die angibt, von welcher Person die Messung stammt
GROUP_COL = "subject"


# X_train enthält nur die echten Sensor-Features.
# Das sind die Eingaben für das Modell.
X_train = train_data[feature_cols]

# y_train enthält die richtigen Aktivitätslabels.
# Das ist das, was das Modell lernen soll.
y_train = train_data[TARGET_COL]

# groups_train enthält die Personen-ID jeder Zeile.
# Das brauchen wir später für GroupKFold.
groups_train = train_data[GROUP_COL]


# X_test enthält die Sensor-Features des finalen Testsets.
X_test = test_data[feature_cols]

# y_test enthält die richtigen Labels des finalen Testsets.
y_test = test_data[TARGET_COL]

# groups_test enthält die Personen-IDs im finalen Testset.
groups_test = test_data[GROUP_COL]

In [9]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("groups_train:", groups_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)
print("groups_test:", groups_test.shape)

print("\nIst 'subject' in X_train?", "subject" in X_train.columns)
print("Ist 'activity' in X_train?", "activity" in X_train.columns)

print("\nTrain subjects:", sorted(groups_train.unique()))
print("Test subjects:", sorted(groups_test.unique()))

print("\nActivity distribution train:")
print(y_train.value_counts())

print("\nActivity distribution test:")
print(y_test.value_counts())

X_train: (5867, 561)
y_train: (5867,)
groups_train: (5867,)
X_test: (1485, 561)
y_test: (1485,)
groups_test: (1485,)

Ist 'subject' in X_train? False
Ist 'activity' in X_train? False

Train subjects: [np.float64(1.0), np.float64(3.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(11.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(19.0), np.float64(21.0), np.float64(22.0), np.float64(23.0), np.float64(25.0), np.float64(26.0)]
Test subjects: [np.float64(27.0), np.float64(28.0), np.float64(29.0), np.float64(30.0)]

Activity distribution train:
activity
LAYING                1114
STANDING              1091
SITTING               1022
WALKING                997
WALKING_UPSTAIRS       857
WALKING_DOWNSTAIRS     786
Name: count, dtype: int64

Activity distribution test:
activity
LAYING                293
STANDING              283
SITTING               264
WALKING               229
WALKING_UPSTAIRS      216
WALKING_DOWNSTAI

# Cross Validation

Objekt und Bewertungsmetriken erstellen

In [10]:
cv = GroupKFold(n_splits=5)

scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro"
}

In [12]:
log_reg_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        C=1.0,
        random_state=42
    ))
])

models = {
    "Logistic Regression": log_reg_model,
}

In [13]:
def evaluate_model_cv(model_name, model, X, y, groups, cv, scoring):
    start_time = time.time()
    
    scores = cross_validate(
        estimator=model,
        X=X,
        y=y,
        groups=groups,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )
    
    total_time = time.time() - start_time
    
    return {
        "model": model_name,
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "f1_macro_mean": scores["test_f1_macro"].mean(),
        "f1_macro_std": scores["test_f1_macro"].std(),
        "fit_time_mean": scores["fit_time"].mean(),
        "score_time_mean": scores["score_time"].mean(),
        "total_time": total_time
    }

verschiedene parameter mit Logistic-Regeression

In [16]:
models = {
    "LogReg C=0.01": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=5000,
            solver="lbfgs",
            C=0.01,
            random_state=42
        ))
    ]),
    
    "LogReg C=0.1": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=5000,
            solver="lbfgs",
            C=0.1,
            random_state=42
        ))
    ]),
    
    "LogReg C=1": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=5000,
            solver="lbfgs",
            C=1.0,
            random_state=42
        ))
    ]),
    
    "LogReg C=10": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=5000,
            solver="lbfgs",
            C=10.0,
            random_state=42
        ))
    ]),
    
    "LogReg C=100": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=5000,
            solver="lbfgs",
            C=100.0,
            random_state=42
        ))
    ]),
}

Auswertung/Cross Validation

In [17]:
results = []

for model_name, model in models.items():
    print(f"Evaluating: {model_name}")
    
    result = evaluate_model_cv(
        model_name=model_name,
        model=model,
        X=X_train,
        y=y_train,
        groups=groups_train,
        cv=cv,
        scoring=scoring
    )
    
    results.append(result)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="accuracy_mean", ascending=False)

display(results_df)

Evaluating: LogReg C=0.01
Evaluating: LogReg C=0.1
Evaluating: LogReg C=1
Evaluating: LogReg C=10
Evaluating: LogReg C=100


,model,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std,fit_time_mean,score_time_mean,total_time
1,LogReg C=0.1,0.915994,0.058786,0.913188,0.055669,1.060008,0.022843,3.687674
2,LogReg C=1,0.915274,0.060308,0.912913,0.057117,1.386972,0.023080,2.112775
0,LogReg C=0.01,0.914818,0.050452,0.910697,0.048949,1.904753,0.026223,6.158225
3,LogReg C=10,0.902893,0.059197,0.900707,0.056382,1.599433,0.022370,2.248720
4,LogReg C=100,0.900815,0.054662,0.897746,0.051047,1.099207,0.021342,1.742490


Bei der Logistic Regression wurden verschiedene Regularisierungsstärken getestet. 
Die beste mittlere Cross-Validation-Accuracy wurde mit C=0.1 erreicht. 
Da der Unterschied zu C=1.0 sehr klein war, deutet dies darauf hin, dass die Logistic Regression relativ stabil gegenüber moderater Regularisierung ist. 
Zu große C-Werte verschlechterten die Leistung leicht, was auf Overfitting bzw. zu schwache Regularisierung hindeuten kann.

In [18]:
best_log_reg_model = models["LogReg C=0.1"]
best_log_reg_name = "LogReg C=0.1"

print("Best Logistic Regression model:", best_log_reg_name)

Best Logistic Regression model: LogReg C=0.1


# RBF SVM

Als nächstes wird eine Support Vector Machine mit RBF-Kernel getestet.

Im Gegensatz zur Logistic Regression ist dieses Modell nichtlinear.  
Es kann dadurch komplexere Entscheidungsgrenzen lernen, ist aber oft rechenintensiver und stärker von den Hyperparametern `C` und `gamma` abhängig.